# 多探针宇宙学约束

本教程演示 HIcosmo 的多探针联合约束分析。

**关键 API**：
- `sne + bao + cmb` — 联合似然（+ 运算符）
- `Plotter([chains], labels)` — 多链比较可视化
- `information_criteria()` — 模型选择（AIC/BIC）
- `plotter.report()` — 统计报告

In [ ]:
import hicosmo as hc
hc.init()

## 1. 创建似然

In [ ]:
from hicosmo.samplers import MCMC
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood, Planck
from hicosmo.models import LCDM, wCDM, CPL

# 创建似然
sne = SN_likelihood(LCDM, "pantheon+")
bao = BAO_likelihood(LCDM, "desi2024")
cmb = Planck(LCDM)

# 联合似然
joint = sne + bao + cmb

print(f"SNe: {sne.n_sne} supernovae")
print(f"BAO: DESI 2024")
print(f"CMB: Planck 2018")

## 2. LCDM 多探针约束

In [ ]:
params = {
    'H0': (68.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

# 各探针单独约束
mcmc_sne = MCMC(params, sne, chain_name='mp_sne')
mcmc_sne.run(num_samples=2000)

mcmc_bao = MCMC(params, bao, chain_name='mp_bao')
mcmc_bao.run(num_samples=2000)

# 联合约束
mcmc_joint = MCMC(params, joint, chain_name='mp_joint')
samples_joint = mcmc_joint.run(num_samples=3000)
mcmc_joint.print_summary()

In [ ]:
from hicosmo.visualization import Plotter

# 多探针比较
plotter = Plotter(
    ['mp_sne', 'mp_bao', 'mp_joint'],
    labels=['Pantheon+ SNe', 'DESI BAO', 'Joint']
)
plotter.corner(['H0', 'Omega_m'], filename='figures/10_lcdm_multiprobe.pdf')

## 3. wCDM 约束

In [ ]:
# wCDM 似然
sne_w = SN_likelihood(wCDM, "pantheon+")
bao_w = BAO_likelihood(wCDM, "desi2024")
cmb_w = Planck(wCDM)
joint_w = sne_w + bao_w + cmb_w

params_w = {
    'H0': (68.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
    'w': (-1.0, -2.0, 0.0),
}

mcmc_wcdm = MCMC(params_w, joint_w, chain_name='mp_wcdm')
samples_wcdm = mcmc_wcdm.run(num_samples=3000)
mcmc_wcdm.print_summary()

In [ ]:
# w-Omega_m 约束
import numpy as np

w_mean = np.mean(samples_wcdm['w'])
w_std = np.std(samples_wcdm['w'])
print(f"w = {w_mean:.3f} ± {w_std:.3f}")
print(f"与 LCDM (w=-1) 偏离: {abs(w_mean + 1) / w_std:.2f}σ")

## 4. CPL 约束

In [ ]:
# CPL: w(z) = w0 + wa × z/(1+z)
sne_cpl = SN_likelihood(CPL, "pantheon+")
bao_cpl = BAO_likelihood(CPL, "desi2024")
cmb_cpl = Planck(CPL)
joint_cpl = sne_cpl + bao_cpl + cmb_cpl

params_cpl = {
    'H0': (68.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
    'w0': (-1.0, -2.0, 0.0),
    'wa': (0.0, -3.0, 2.0),
}

mcmc_cpl = MCMC(params_cpl, joint_cpl, chain_name='mp_cpl')
samples_cpl = mcmc_cpl.run(num_samples=2000)
mcmc_cpl.print_summary()

In [ ]:
# w0-wa 约束与 Figure of Merit
w0 = samples_cpl['w0']
wa = samples_cpl['wa']
cov = np.cov(w0, wa)
FoM = 1.0 / np.sqrt(np.linalg.det(cov))

print(f"w0 = {np.mean(w0):.3f} ± {np.std(w0):.3f}")
print(f"wa = {np.mean(wa):.3f} ± {np.std(wa):.3f}")
print(f"Figure of Merit = {FoM:.1f}")

## 5. 模型比较

In [ ]:
from hicosmo.visualization import information_criteria

num_data = sne.n_sne + 12 + 3  # SNe + BAO + CMB

# LCDM
ic_lcdm = information_criteria(
    samples=samples_joint,
    log_likelihood_fn=joint,
    num_data=num_data,
    param_names=['H0', 'Omega_m']
)

# wCDM
ic_wcdm = information_criteria(
    samples=samples_wcdm,
    log_likelihood_fn=joint_w,
    num_data=num_data,
    param_names=['H0', 'Omega_m', 'w']
)

# CPL
ic_cpl = information_criteria(
    samples=samples_cpl,
    log_likelihood_fn=joint_cpl,
    num_data=num_data,
    param_names=['H0', 'Omega_m', 'w0', 'wa']
)

print(f"{'Model':<10} {'AIC':<12} {'BIC':<12} {'ΔBIC':<10}")
print("-" * 44)
for name, ic in [('LCDM', ic_lcdm), ('wCDM', ic_wcdm), ('CPL', ic_cpl)]:
    delta = ic['bic'] - ic_lcdm['bic']
    print(f"{name:<10} {ic['aic']:<12.2f} {ic['bic']:<12.2f} {delta:<+10.2f}")

## 6. 模型对比可视化

In [ ]:
# 所有模型对比
plotter_models = Plotter(
    ['mp_joint', 'mp_wcdm', 'mp_cpl'],
    labels=['LCDM', 'wCDM', 'CPL']
)
plotter_models.corner(['H0', 'Omega_m'], filename='figures/10_model_comparison.pdf')

## 7. Hubble 张力

In [ ]:
# Hubble 张力分析
H0_our = np.mean(samples_joint['H0'])
H0_err = np.std(samples_joint['H0'])
H0_shoes, H0_shoes_err = 73.04, 1.04  # SH0ES 2022

tension = abs(H0_our - H0_shoes) / np.sqrt(H0_err**2 + H0_shoes_err**2)

print(f"本分析: H0 = {H0_our:.2f} ± {H0_err:.2f} km/s/Mpc")
print(f"SH0ES:  H0 = {H0_shoes:.2f} ± {H0_shoes_err:.2f} km/s/Mpc")
print(f"张力: {tension:.1f}σ")

## API 速查

```python
from hicosmo.samplers import MCMC
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood, Planck
from hicosmo.models import LCDM, wCDM, CPL
from hicosmo.visualization import Plotter, information_criteria

# === 创建似然 ===
sne = SN_likelihood(LCDM, "pantheon+")
bao = BAO_likelihood(LCDM, "desi2024")
cmb = Planck(LCDM)
joint = sne + bao + cmb  # + 运算符组合

# === MCMC 约束 ===
params = {'H0': (68, 60, 80), 'Omega_m': (0.3, 0.1, 0.5)}
mcmc = MCMC(params, joint, chain_name='analysis')
samples = mcmc.run(num_samples=3000)

# === 多链比较 ===
plotter = Plotter(['chain1', 'chain2'], labels=['Probe A', 'Probe B'])
plotter.corner(['H0', 'Omega_m'], filename='comparison.pdf')

# === 模型选择 ===
ic = information_criteria(
    samples=samples,
    log_likelihood_fn=likelihood,
    num_data=N,
    param_names=['H0', 'Omega_m']
)
print(f"AIC={ic['aic']}, BIC={ic['bic']}")
```

### 探针组合

| 组合 | 优势 |
|------|------|
| SNe | 测量 d_L(z)，约束 H0 |
| BAO | 测量 D_A/r_d, H·r_d，约束 Ω_m |
| CMB | 测量早期宇宙距离，高精度 |
| SNe+BAO+CMB | 打破简并，最强约束 |

### Jeffreys ΔBIC 尺度

| ΔBIC | 解读 |
|------|------|
| < 2 | 无显著差异 |
| 2-6 | 正证据 |
| 6-10 | 强证据 |
| > 10 | 决定性证据 |